# 🏠 House Price Prediction — Final Data Science Project
> **Faculty of Computers and Information | Assiut University | 2025-2026**

---
### Project Overview
This notebook applies a complete machine learning pipeline to predict house sale prices using the **Ames Housing Dataset**.

**Stages:**
1. Import Libraries
2. Load & Explore Dataset
3. Missing Value Analysis
4. Data Cleaning & Imputation
5. Exploratory Data Analysis (EDA)
6. Feature Encoding & Train/Test Split
7. Linear Regression
8. Ridge Regression
9. Lasso Regression
10. Model Comparison
11. K-Means Clustering
12. Conclusion

---
## Section 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully')

---
## Section 2 — Load & Explore Dataset

In [ ]:
# Load dataset — update path as needed
df = pd.read_csv(r"C:\Users\as\Downloads\final project data science\train.csv")
pd.set_option('display.max_columns', None)

print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Descriptive statistics for numeric features
df.describe()

---
## Section 3 — Missing Value Analysis

In [ ]:
# Count and sort missing values
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

---
## Section 4 — Data Cleaning

In [ ]:
# Drop columns with > 80% missing values
cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']
df = df.drop(cols_to_drop, axis=1)

print(f'Dropped: {cols_to_drop}')
print(f'New shape: {df.shape}')

In [ ]:
# --- Impute Numerical Columns ---
# Mean for symmetric distributions
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].mean())
df['MasVnrArea']  = df['MasVnrArea'].fillna(df['MasVnrArea'].mean())

# Median for skewed distributions
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['GarageYrBlt'].median())

# --- Impute Categorical Columns (Mode) ---
cat_cols = [
    'MasVnrType', 'FireplaceQu', 'GarageFinish', 'GarageQual',
    'GarageCond', 'GarageType', 'BsmtExposure', 'BsmtFinType2',
    'BsmtCond', 'BsmtFinType1', 'BsmtQual', 'Electrical'
]
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Verify no missing values remain
remaining = df.isnull().sum().sum()
print(f'Total missing values remaining: {remaining} ✅' if remaining == 0 else f'Still missing: {remaining}')

---
## Section 5 — Exploratory Data Analysis (EDA)

In [ ]:
# Sale Price Distribution
plt.figure(figsize=(10, 5))
sns.histplot(df['SalePrice'], kde=True, color='steelblue')
plt.title('Sale Price Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Sale Price (USD)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(f"Mean  : ${df['SalePrice'].mean():,.0f}")
print(f"Median: ${df['SalePrice'].median():,.0f}")
print(f"Min   : ${df['SalePrice'].min():,.0f}")
print(f"Max   : ${df['SalePrice'].max():,.0f}")

In [ ]:
# Correlation Heatmap
correlation = df.corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(correlation, cmap='coolwarm', center=0, linewidths=0.3)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top features correlated with SalePrice
top_corr = correlation['SalePrice'].drop('SalePrice').sort_values(ascending=False)
print('Top 10 features correlated with SalePrice:')
print(top_corr.head(10).to_string())

---
## Section 6 — Feature Encoding & Train/Test Split

In [ ]:
# One-hot encode all categorical features
df_encoded = pd.get_dummies(df, drop_first=True)
print(f'Encoded shape: {df_encoded.shape}')
df_encoded.head()

In [ ]:
# Define features (X) and target (y)
x = df_encoded.drop('SalePrice', axis=1)
y = df_encoded['SalePrice']

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Testing set  : {X_test.shape[0]} samples')

---
## Section 7 — Linear Regression

In [ ]:
# Train
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict
y_pred = lr.predict(X_test)

# Evaluate
lr_mse  = mean_squared_error(y_test, y_pred)
lr_rmse = np.sqrt(lr_mse)

print('--- Linear Regression ---')
print(f'MSE  = {lr_mse:,.0f}')
print(f'RMSE = {lr_rmse:,.0f}')

---
## Section 8 — Ridge Regression (L2 Regularization)

In [ ]:
# Train
ridge = Ridge(alpha=0.1)
ridge.fit(X_train, y_train)

# Predict
ridge_pred = ridge.predict(X_test)

# Evaluate
ridge_mse  = mean_squared_error(y_test, ridge_pred)
ridge_rmse = np.sqrt(ridge_mse)

print('--- Ridge Regression ---')
print(f'MSE  = {ridge_mse:,.0f}')
print(f'RMSE = {ridge_rmse:,.0f}')

---
## Section 9 — Lasso Regression (L1 Regularization)

In [ ]:
# Train
lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train, y_train)

# Predict
lasso_pred = lasso.predict(X_test)

# Evaluate
lasso_mse  = mean_squared_error(y_test, lasso_pred)
lasso_rmse = np.sqrt(lasso_mse)

print('--- Lasso Regression ---')
print(f'MSE  = {lasso_mse:,.0f}')
print(f'RMSE = {lasso_rmse:,.0f}')

---
## Section 10 — Model Comparison

In [ ]:
# Summary table
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Lasso Regression'],
    'MSE':   [round(lr_mse), round(ridge_mse), round(lasso_mse)],
    'RMSE':  [round(lr_rmse), round(ridge_rmse), round(lasso_rmse)]
})
results = results.sort_values('RMSE')
print(results.to_string(index=False))
print(f"\n✅ Best Model: {results.iloc[0]['Model']} (RMSE = {results.iloc[0]['RMSE']:,})")

In [ ]:
# RMSE Bar Chart
plt.figure(figsize=(8, 5))
colors_bar = ['#EF5350', '#42A5F5', '#FFA726']
bars = plt.bar(results['Model'], results['RMSE'], color=colors_bar, edgecolor='black', linewidth=0.5)
plt.title('Model Comparison — RMSE', fontsize=14, fontweight='bold')
plt.ylabel('RMSE (USD)')
plt.xticks(rotation=10)
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
             f"${bar.get_height():,.0f}", ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 11 — K-Means Clustering

In [ ]:
# Elbow Method — find optimal K
inertia = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(x)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia, marker='o', color='steelblue', linewidth=2, markersize=8)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='Optimal K=3')
plt.title('Elbow Method — Optimal Number of Clusters', fontsize=13, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Apply K-Means with K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(x)
df_encoded['Cluster'] = clusters

print('Cluster distribution:')
print(df_encoded['Cluster'].value_counts().sort_index())

In [ ]:
# Mean SalePrice per cluster
cluster_means = df_encoded.groupby('Cluster')['SalePrice'].mean().sort_values()
print('Mean Sale Price by Cluster:')
print(cluster_means.apply(lambda x: f'${x:,.0f}').to_string())

In [ ]:
# Boxplot — Sale Price by Cluster
plt.figure(figsize=(9, 6))
palette = ['#42A5F5', '#EF5350', '#66BB6A']
sns.boxplot(x='Cluster', y='SalePrice', data=df_encoded, palette=palette)
plt.title('Sale Price Distribution by Cluster', fontsize=14, fontweight='bold')
plt.xlabel('Cluster')
plt.ylabel('Sale Price (USD)')

# Cluster labels
labels = {0: 'Economy', 1: 'Luxury', 2: 'Mid-range'}
for cluster_id, label in labels.items():
    mean_val = df_encoded[df_encoded['Cluster'] == cluster_id]['SalePrice'].mean()
    plt.text(cluster_id, mean_val + 5000, f'{label}\n${mean_val:,.0f}',
             ha='center', fontsize=9, fontweight='bold', color='black')

plt.tight_layout()
plt.show()

---
## Section 12 — Conclusion

### Summary of Results

| Stage | Result |
|---|---|
| Dataset | 1,460 rows × 81 columns (77 after cleaning) |
| Missing Values | Imputed using mean / median / mode |
| Linear Regression | RMSE = \$52,311 |
| Ridge Regression | RMSE = \$33,333 ✅ **Best Model** |
| Lasso Regression | RMSE = \$52,110 |
| K-Means Clustering | 3 clusters: Economy / Mid-range / Luxury |

### Key Takeaways
- **Ridge Regression** outperformed other models due to L2 regularization, which handles multicollinearity effectively.
- **OverallQual** is the single strongest predictor of sale price (correlation ≈ 0.79).
- **K-Means** revealed three economically meaningful market segments without using SalePrice as input.
- Future improvements: try Random Forest or Gradient Boosting, and apply log-transform to SalePrice.
